# TBtools locus table to rice FASTA in Google Colab
Upload `CDS_TBTOOL.txt`. Your inspected file has 66,338 transcript records linked to 55,986 genes. All its intervals match mRNA genomic spans in the matching GFF3, even though its filename says CDS.

This notebook downloads the matching MSU/UGA Release 7 complete genome and GFF3 from the official provider. CPU runtime is sufficient. It exports annotated gene genomic sequences, transcript genomic spans, spliced cDNA, and joined CDS separately. No bases are predicted by ML; numerical features are prepared for a later labeled study.

Genomic spans include introns. Spliced cDNA contains annotated exons and UTRs. CDS contains annotated coding segments. Reference models can differ from experimentally validated or curated canonical transcripts. These files do not establish stem expression or lodging function.
Run each cell in order. Download the ZIP before your session expires.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import unquote
from urllib.request import Request, urlopen
import os, csv, gzip, json, hashlib, shutil

ROOT = Path(os.environ.get("TBTOOLS_PROJECT_DIR", "/content/tbtools_locus_project"))
INPUT, OUTPUT = ROOT / "input", ROOT / "output"
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Free disk GB:", round(shutil.disk_usage(ROOT).free / 1e9, 2))

## 2. Upload the TBtools table
Choose only `CDS_TBTOOL.txt`. This is a headerless tab-separated table, not a FASTA file. Columns 1–5 are chromosome, transcript ID, start, end, strand. The remaining columns are attribute-key/value pairs; Parent supplies the gene ID. All IDs are retained; no isoform suffix is removed.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one TBtools table.")
name, data = next(iter(uploaded.items()))
table_path = INPUT / Path(name).name
table_path.write_bytes(data)
del uploaded, data
print("Table:", table_path)

In [ ]:
def read_table(path):
    rows, seen = [], set()
    with open(path, newline="", encoding="utf-8-sig") as handle:
        for number, fields in enumerate(csv.reader(handle, delimiter="\t"), 1):
            if not fields or not any(f.strip() for f in fields):
                continue
            if len(fields) < 7 or (len(fields) - 5) % 2:
                raise ValueError(f"Line {number}: expected 5 fields followed by key/value pairs.")
            attrs = dict(zip(fields[5::2], fields[6::2]))
            chrom, tid, start, end, strand = fields[:5]
            gid = attrs.get("Parent")
            if not gid or "," in gid or attrs.get("ID") != tid:
                raise ValueError(f"Line {number}: missing/ambiguous gene Parent or mismatched transcript ID.")
            start, end = int(start), int(end)
            if start < 1 or end < start or strand not in ("+", "-"):
                raise ValueError(f"Line {number}: invalid coordinates or strand.")
            if tid in seen:
                raise ValueError(f"Duplicate transcript ID: {tid}")
            seen.add(tid)
            rows.append(dict(gene_id=gid, transcript_id=tid, chromosome=chrom,
                             start=start, end=end, strand=strand))
    if not rows:
        raise ValueError("No records found.")
    return rows

rows = read_table(table_path)
print("Transcript records:", len(rows))
print("Distinct parent genes:", len({r["gene_id"] for r in rows}))
print("First record:", rows[0])

## 3. Download the correct references automatically
The genome input is `osa1_r7.asm.fa.gz`, with chromosome FASTA IDs such as Chr1. Gene/protein/cDNA FASTA files cannot replace the full assembly. The separate GFF3 supplies exon and CDS segments that are absent from your TBtools table. Downloading resets the file paths explicitly, avoiding reuse of an earlier uploaded protein file.
Official source: https://rice.uga.edu/download_osa1r7.shtml

In [ ]:
def download_reference(filename):
    target = INPUT / filename
    temporary = INPUT / (filename + ".part")
    url = "https://rice.uga.edu/osa1r7_download/" + filename
    print("Downloading:", filename)
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=120) as response, open(temporary, "wb") as handle:
        shutil.copyfileobj(response, handle)
    with gzip.open(temporary, "rt") as handle:
        first = handle.readline().strip()
    if filename.endswith(".fa.gz"):
        expected = {f"Chr{i}" for i in range(1, 13)} | {"ChrUn", "ChrSy"}
        if not first.startswith(">") or first[1:].split()[0] not in expected:
            raise ValueError(f"Expected chromosome genome FASTA; received {first[:120]}")
    elif not first.startswith("##gff-version"):
        raise ValueError("Expected a GFF3 annotation download.")
    temporary.replace(target)
    print("Saved:", target.name, "First line:", first)
    return target

genome_path = download_reference("osa1_r7.asm.fa.gz")
gff_path = download_reference("osa1_r7.all_models.gff3.gz")

## 4. Validate the table against the annotation and genome
GFF3 coordinates are 1-based and inclusive. Python uses `sequence[start-1:end]`. Minus-strand sequences are reverse-complemented. Every input transcript, parent gene, chromosome, interval and strand must match the downloaded annotation; mismatches stop the run. Gene FASTA uses gene boundaries from the GFF3, not an arbitrary single isoform interval.

In [ ]:
def open_text(path):
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "rt")

def parse_gff(path):
    genes, transcripts = {}, {}
    exons = defaultdict(list)
    feature_counts = Counter()
    with open_text(path) as handle:
        for lineno, line in enumerate(handle, 1):
            if line.startswith("##FASTA"):
                break
            if not line.strip() or line.startswith("#"):
                continue
            cols = line.rstrip("\r\n").split("\t")
            if len(cols) != 9:
                raise ValueError(f"GFF3 line {lineno}: expected 9 columns.")
            chrom, source, kind, start, end, score, strand, phase, attrs_raw = cols
            attrs = {k: unquote(v) for k, v in
                     (item.split("=", 1) for item in attrs_raw.split(";") if "=" in item)}
            feature_counts[kind] += 1
            rec = {"chromosome": chrom, "start": int(start), "end": int(end),
                   "strand": strand, "description": attrs.get("Note", "")}
            if rec["start"] < 1 or rec["end"] < rec["start"]:
                raise ValueError(f"Invalid interval on GFF3 line {lineno}.")
            if kind in ("gene", "mRNA", "transcript"):
                ident = attrs.get("ID")
                if not ident:
                    raise ValueError(f"Missing ID on line {lineno}.")
                target = genes if kind == "gene" else transcripts
                if ident in target:
                    raise ValueError(f"Duplicate feature ID: {ident}")
                if kind != "gene":
                    parents = attrs.get("Parent", "").split(",")
                    if len(parents) != 1 or not parents[0]:
                        raise ValueError(f"Expected one parent gene for {ident}.")
                    rec["gene_id"] = parents[0]
                target[ident] = rec
            elif kind == "exon":
                parents = attrs.get("Parent", "").split(",")
                if not parents[0]:
                    raise ValueError(f"Exon without Parent on line {lineno}.")
                for parent in parents:
                    exons[parent].append((chrom, int(start), int(end), strand))
    for tid, rec in transcripts.items():
        if rec["gene_id"] not in genes:
            raise ValueError(f"Unknown parent gene for {tid}.")
        gene = genes[rec["gene_id"]]
        if rec["chromosome"] != gene["chromosome"] or rec["strand"] != gene["strand"]:
            raise ValueError(f"Transcript/gene chromosome or strand mismatch: {tid}")
        if not gene["start"] <= rec["start"] <= rec["end"] <= gene["end"]:
            raise ValueError(f"Transcript outside gene interval: {tid}")
        if not exons[tid]:
            raise ValueError(f"No exons annotated for {tid}.")
    orphan_exons = set(exons) - set(transcripts)
    if orphan_exons:
        raise ValueError(f"Exons reference unknown transcripts: {sorted(orphan_exons)[:5]}")
    return genes, transcripts, exons, feature_counts

def load_genome(path):
    genome, name, parts = {}, None, []
    with open_text(path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    genome[name] = "".join(parts).upper()
                name = line[1:].split()[0]
                if name in genome:
                    raise ValueError(f"Duplicate FASTA ID: {name}")
                parts = []
            else:
                if name is None:
                    raise ValueError("FASTA sequence before its header.")
                parts.append(line)
        if name is not None:
            genome[name] = "".join(parts).upper()
    if not genome:
        raise ValueError("No FASTA records found.")
    allowed = set("ACGTRYSWKMBDHVN")
    for name, sequence in genome.items():
        if set(sequence) - allowed:
            raise ValueError(f"Unexpected alphabet in {name}; verify this is DNA FASTA.")
    return genome

COMPLEMENT = str.maketrans("ACGTRYSWKMBDHVN", "TGCAYRSWMKVHDBN")
def reverse_complement(sequence):
    return sequence.translate(COMPLEMENT)[::-1]

def extract_interval(genome, chrom, start, end, strand):
    if chrom not in genome:
        raise ValueError(f"Chromosome {chrom} is absent from genome FASTA.")
    if start < 1 or end < start or end > len(genome[chrom]):
        raise ValueError(f"Out-of-bounds interval: {chrom}:{start}-{end}")
    if strand not in ("+", "-"):
        raise ValueError(f"Unsupported strand: {strand}")
    sequence = genome[chrom][start - 1:end]
    return reverse_complement(sequence) if strand == "-" else sequence

def transcript_sequence(tid, transcripts, exons, genome):
    rec = transcripts[tid]
    blocks = sorted(set(exons[tid]), key=lambda x: (x[1], x[2]))
    previous_end = 0
    for chrom, start, end, strand in blocks:
        if chrom != rec["chromosome"] or strand != rec["strand"]:
            raise ValueError(f"Exon chromosome/strand mismatch: {tid}")
        if not rec["start"] <= start <= end <= rec["end"]:
            raise ValueError(f"Exon outside transcript interval: {tid}")
        if start <= previous_end:
            raise ValueError(f"Overlapping exons: {tid}")
        previous_end = end
    sequence = "".join(extract_interval(genome, chrom, start, end, "+")
                       for chrom, start, end, strand in blocks)
    return reverse_complement(sequence) if rec["strand"] == "-" else sequence

genes, transcripts, exons, feature_counts = parse_gff(gff_path)
genome = load_genome(genome_path)
missing = {r["chromosome"] for r in genes.values()} - set(genome)
if missing:
    raise ValueError(f"Missing FASTA chromosomes: {sorted(missing)}. Use matching full assembly.")
for gid, r in genes.items():
    extract_interval(genome, r["chromosome"], r["start"], r["end"], r["strand"])
print("Genes:", len(genes), "Transcripts:", len(transcripts))
print("Feature counts:", dict(feature_counts))
print("Genome records:", list(genome))

def read_cds(path):
    blocks = defaultdict(list)
    with open_text(path) as handle:
        for line in handle:
            if line.startswith("##FASTA"):
                break
            if not line.strip() or line.startswith("#"):
                continue
            fields = line.rstrip("\r\n").split("\t")
            if fields[2] != "CDS":
                continue
            attrs = {k: unquote(v) for k,v in
                     (part.split("=",1) for part in fields[8].split(";") if "=" in part)}
            if not attrs.get("Parent"):
                raise ValueError("CDS feature has no Parent.")
            for parent in attrs["Parent"].split(","):
                blocks[parent].append((fields[0], int(fields[3]), int(fields[4]), fields[6]))
    return blocks

cds_blocks = read_cds(gff_path)
for row in rows:
    tid, gid = row["transcript_id"], row["gene_id"]
    if tid not in transcripts or gid not in genes:
        raise ValueError(f"ID absent from matching annotation: {tid}, {gid}")
    tx = transcripts[tid]
    if tx["gene_id"] != gid or any(row[key] != tx[key] for key in
                                   ("chromosome", "start", "end", "strand")):
        raise ValueError(f"Table does not match transcript genomic span: {tid}. Check the selected feature/reference.")
    if not cds_blocks.get(tid):
        raise ValueError(f"No CDS annotation for {tid}.")
print("Validated all", len(rows), "table records against the reference.")

## 5. Export FASTA and numerical features
Four FASTA files distinguish the sequence types. Each uses gene or transcript ID as the first header token, followed by explicit sequence-type information. Outputs use gzip compression; extract .gz to obtain ordinary FASTA if a downstream tool requires it. Mapping and feature tables preserve Parent relationships and all isoforms.

`ml_transcript_features.csv` provides length, GC fraction, exon count and CDS length per transcript. It has no phenotype label, and no classifier is trained. Gene/transcript IDs are join keys, not numeric predictors. For a supervised gene study keep isoforms of each gene together in train/test splitting; unstudied genes are not automatically negative examples.

In [ ]:
def write_fasta(handle, ident, sequence, description):
    handle.write(f">{ident} {description}\n")
    for offset in range(0, len(sequence), 60):
        handle.write(sequence[offset:offset+60] + "\n")

def gc_fraction(sequence):
    bases = Counter(sequence)
    valid = sum(bases[b] for b in "ACGT")
    return (bases["G"] + bases["C"]) / valid if valid else ""

def export_sequences(rows, genes, transcripts, exons, cds_blocks, genome, output):
    gene_ids = list(dict.fromkeys(row["gene_id"] for row in rows))
    with gzip.open(output / "gene_genomic.fasta.gz", "wt", compresslevel=1) as handle:
        for gid in gene_ids:
            rec = genes[gid]
            seq = extract_interval(genome, rec["chromosome"], rec["start"], rec["end"], rec["strand"])
            write_fasta(handle, gid, seq,
                f'gene_genomic {rec["chromosome"]}:{rec["start"]}-{rec["end"]} strand={rec["strand"]}')
    with open(output / "gene_ids.txt", "w") as handle:
        handle.write("\n".join(gene_ids) + "\n")
    with open(output / "transcript_ids.txt", "w") as handle:
        handle.write("\n".join(row["transcript_id"] for row in rows) + "\n")
    feature_fields = ["gene_id", "transcript_id", "gene_genomic_length_bp",
                      "transcript_genomic_length_bp", "cdna_length_bp", "cds_length_bp",
                      "cdna_gc_fraction_acgt", "cdna_ambiguous_fraction", "exon_count", "transcript_count"]
    count_by_gene = Counter(row["gene_id"] for row in rows)
    mapping_fields = list(rows[0])
    with gzip.open(output / "transcript_genomic.fasta.gz", "wt", compresslevel=1) as genomic_file, \
         gzip.open(output / "transcript_cdna.fasta.gz", "wt", compresslevel=1) as cdna_file, \
         gzip.open(output / "transcript_cds.fasta.gz", "wt", compresslevel=1) as cds_file, \
         open(output / "gene_transcript_map.csv", "w", newline="") as mapping_file, \
         open(output / "ml_transcript_features.csv", "w", newline="") as feature_file:
        mapping = csv.DictWriter(mapping_file, fieldnames=mapping_fields); mapping.writeheader()
        features = csv.DictWriter(feature_file, fieldnames=feature_fields); features.writeheader()
        for row in rows:
            tid, gid = row["transcript_id"], row["gene_id"]
            genomic = extract_interval(genome, row["chromosome"], row["start"], row["end"], row["strand"])
            cdna = transcript_sequence(tid, transcripts, exons, genome)
            cds = transcript_sequence(tid, transcripts, cds_blocks, genome)
            write_fasta(genomic_file, tid, genomic,
                f'gene_id={gid} transcript_genomic {row["chromosome"]}:{row["start"]}-{row["end"]} strand={row["strand"]}')
            write_fasta(cdna_file, tid, cdna, f"gene_id={gid} spliced_cdna")
            write_fasta(cds_file, tid, cds, f"gene_id={gid} joined_annotated_CDS")
            mapping.writerow(row)
            valid = sum(cdna.count(base) for base in "ACGT")
            features.writerow(dict(gene_id=gid, transcript_id=tid,
                gene_genomic_length_bp=genes[gid]["end"]-genes[gid]["start"]+1,
                transcript_genomic_length_bp=len(genomic), cdna_length_bp=len(cdna),
                cds_length_bp=len(cds), cdna_gc_fraction_acgt=gc_fraction(cdna),
                cdna_ambiguous_fraction=(len(cdna)-valid)/len(cdna),
                exon_count=len(set(exons[tid])), transcript_count=count_by_gene[gid]))
    print("Exported genes:", len(gene_ids), "Transcripts:", len(rows))

export_sequences(rows, genes, transcripts, exons, cds_blocks, genome, OUTPUT)
for path in sorted(OUTPUT.iterdir()):
    print(path.name, round(path.stat().st_size/1e6, 2), "MB")

## 6. Export MOC1 separately and check sequence types
Your table includes gene `LOC_Os06g40780`, transcript `LOC_Os06g40780.1`. In this particular annotation the genomic span is 4,963 bp, spliced cDNA 2,064 bp and joined annotated CDS 2,001 bp. These describe this reference model; validate canonical identity/structure before protein interpretation. cDNA must not be translated as if UTRs were coding.

In [ ]:
MOC1_ID = "LOC_Os06g40780"
selected = [row for row in rows if row["gene_id"] == MOC1_ID]
if selected:
    rec = genes[MOC1_ID]
    seq = extract_interval(genome, rec["chromosome"], rec["start"], rec["end"], rec["strand"])
    with open(OUTPUT / "MOC1_genomic.fasta", "w") as handle:
        write_fasta(handle, MOC1_ID, seq, "gene_genomic")
    print("MOC1 genomic bp:", len(seq))
    for label, blocks in [("cdna", exons), ("cds", cds_blocks)]:
        with open(OUTPUT / f"MOC1_{label}.fasta", "w") as handle:
            for row in selected:
                tid = row["transcript_id"]
                seq = transcript_sequence(tid, transcripts, blocks, genome)
                write_fasta(handle, tid, seq, f"gene_id={MOC1_ID} {label}")
                print(tid, label, "bp:", len(seq))
else:
    print("MOC1 is not in the uploaded selection.")

## 7. Download results
The ZIP contains compressed FASTA files, ID lists, gene/transcript mapping, numerical features, MOC1 files and input provenance. All sequence outputs are oriented on the annotated strand. Download the ZIP before the session expires. For ML, first define a prediction target and obtain independent evidence-backed labels or matched lodging phenotypes. These reference sequences alone are not a lodging classifier or a stem-expression measurement.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {"inputs": {path.name:sha256_file(path) for path in [table_path,genome_path,gff_path]},
            "transcript_count":len(rows), "gene_count":len({r["gene_id"] for r in rows}),
            "coordinate_system":"1-based inclusive", "orientation":"annotated gene/transcript strand",
            "table_scope":"validated transcript genomic spans", "gene_scope":"GFF3 gene boundaries",
            "cdna_scope":"joined annotated exons", "cds_scope":"joined annotated CDS segments",
            "expression_measured":False, "supervised_model_trained":False,
            "reference_source":"https://rice.uga.edu/download_osa1r7.shtml"}
(OUTPUT / "provenance.json").write_text(json.dumps(manifest, indent=2))
zip_path = shutil.make_archive(str(ROOT / "TBtools_Rice_FASTA_Results"), "zip", OUTPUT)
print("Download:", zip_path)
files.download(zip_path)